In [ ]:
import os
import json
import shutil
import random
import csv
from pathlib import Path

import numpy as np
import torch
from ultralytics import YOLO

# ============================================================================
# IMPORT CENTRALIZED CONFIG - All paths defined in config.py
# ============================================================================
from config import (
    DATA_YAML, YOLOV8S_WEIGHTS, YOLO_FIXED_SEED_DIR,
    SEED, IMGSZ, BATCH, DEVICE, FIXED_SEED_EPOCHS, F1_THRESHOLD,
    CLASS_NAMES
)

# Convert Path objects to strings for compatibility
DATA_YAML = str(DATA_YAML)
SAVE_DIR = str(YOLO_FIXED_SEED_DIR)
WEIGHTS = str(YOLOV8S_WEIGHTS)
EPOCHS = FIXED_SEED_EPOCHS


# UTILS

def set_global_seed(seed: int):
    print(f"[SEED] Using seed = {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_paths():
    os.makedirs(SAVE_DIR, exist_ok=True)
    if not os.path.isfile(WEIGHTS):
        raise FileNotFoundError(
            f"Base weights file not found:\n  {WEIGHTS}\n"
            "Fix the WEIGHTS path in config.py."
        )


def find_best_epoch_by_f1(results_csv_path: str):
    """
    Read Ultralytics results.csv and return (best_epoch, best_f1).
    F1 = 2 * P * R / (P + R), using metrics/precision* and metrics/recall* columns.
    """
    if not os.path.isfile(results_csv_path):
        print("[WARN] results.csv not found:", results_csv_path)
        return None, None

    with open(results_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        headers = reader.fieldnames or []
        print("\n[CSV] headers:", headers)

        # try to find precision/recall columns automatically
        prec_col = None
        rec_col  = None
        for h in headers:
            if h.startswith("metrics/precision"):
                prec_col = h
            if h.startswith("metrics/recall"):
                rec_col = h

        if prec_col is None or rec_col is None:
            print("[WARN] Could not find precision/recall columns in results.csv.")
            print("       Won't select best epoch by F1.")
            return None, None

        print(f"[CSV] Using columns: precision='{prec_col}', recall='{rec_col}'")

        best_epoch = None
        best_f1 = -1.0

        for row in reader:
            try:
                epoch = int(row["epoch"])
                p = float(row[prec_col])
                r = float(row[rec_col])
            except (KeyError, ValueError):
                continue

            f1 = 2 * p * r / (p + r + 1e-9)
            if f1 > best_f1:
                best_f1 = f1
                best_epoch = epoch

    return best_epoch, best_f1



# MAIN TRAINING

def main():
    ensure_paths()
    set_global_seed(SEED)

    # 1) Init model
    print("\n[INIT] Loading base model from:", WEIGHTS)
    model = YOLO(WEIGHTS)

    # 2) Train on train+val (from data.yaml). NO TEST USED HERE.
    print("\n[TRAIN] Starting fixed-seed training (train + val only)...")
    train_res = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=4,
        pretrained=True,
        optimizer="SGD",
        lr0=0.01,
        lrf=0.1,
        momentum=0.937,
        weight_decay=5e-4,
        cos_lr=True,
        warmup_epochs=3,

        # VIN-friendly aug (no flips/mosaic)
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        flipud=0.0,
        fliplr=0.0,
        mosaic=0.0,
        mixup=0.0,
        degrees=0.0,
        shear=0.0,
        translate=0.05,
        scale=0.10,

        rect=True,
        project=SAVE_DIR,
        name="fixed_seed_run",
        patience=20,
        save_period=10,   # save epochXX.pt every 10 epochs
        seed=SEED,
        verbose=True,
    )

    # 3) Paths from this training
    run_dir   = Path(model.trainer.save_dir)
    best_path = Path(model.trainer.best)
    results_csv = run_dir / "results.csv"

    print("\n[TRAIN] Training finished.")
    print("       Run directory          :", run_dir)
    print("       Ultralytics 'best.pt'  :", best_path)
    print("       results.csv            :", results_csv)

    # 4) Find epoch with best F1 from results.csv
    best_epoch, best_f1 = find_best_epoch_by_f1(str(results_csv))
    print(f"\n[ANALYSIS] Best epoch by F1 from results.csv: {best_epoch}, F1 = {best_f1}")

    # 5) Decide which weights to copy
    if best_epoch is not None and best_f1 is not None and best_f1 >= F1_THRESHOLD:
        weights_dir = run_dir / "weights"
        cand1 = weights_dir / f"epoch{best_epoch}.pt"
        cand2 = weights_dir / f"epoch{best_epoch - 1}.pt"

        if cand1.is_file():
            chosen = cand1
        elif cand2.is_file():
            chosen = cand2
        else:
            print("[WARN] Could not find epoch weights for best_epoch; falling back to best.pt")
            chosen = best_path

        final_ckpt = Path(SAVE_DIR) / "vin_yolov8s_best_fixed_seed.pt"
        shutil.copy(str(chosen), str(final_ckpt))
        print(f"\n[WEIGHTS] F1 >= {F1_THRESHOLD:.3f} → saving epoch {best_epoch} weights as:")
        print("  ", final_ckpt)
    else:
        final_ckpt = Path(SAVE_DIR) / "vin_yolov8s_fixed_seed.pt"
        shutil.copy(str(best_path), str(final_ckpt))
        print(f"\n[WEIGHTS] F1 never reached {F1_THRESHOLD:.3f}.")
        print("          Saving Ultralytics best.pt as:", final_ckpt)

    # 6) Save JSON summary (convert everything to plain types)
    summary = {
        "seed": int(SEED),
        "epochs": int(EPOCHS),
        "best_epoch_by_f1": None if best_epoch is None else int(best_epoch),
        "best_f1": None if best_f1 is None else float(best_f1),
        "f1_threshold": float(F1_THRESHOLD),
        "run_dir": str(run_dir),
        "chosen_weights": str(final_ckpt),
    }
    summary_path = Path(SAVE_DIR) / "fixed_seed_training_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print("\n[SUMMARY] JSON saved to:", summary_path)

    print("\nDone. You now have a fixed-seed trained model ready for later TEST evaluation.")


if __name__ == "__main__":
    main()

[SEED] Using seed = 12345

[INIT] Loading base model from: C:\JLR_VIN_2549\yolov8s.pt

[TRAIN] Starting fixed-seed training (train + val only)...
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\JLR_VIN_2549\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\JLR_VIN_2549\yolov8s.pt, momentum=0.937, mosai

In [ ]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

# Import centralized config
from config import DATA_YAML, YOLO_BEST_FIXED_SEED, TEST_IMAGES

# Load the trained VIN model
model = YOLO(str(YOLO_BEST_FIXED_SEED))

# Evaluate on TEST dataset
metrics = model.val(
    data=str(DATA_YAML),  # dataset config YAML
    split="test"
)

# Show per-class metrics in dataframe
class_metrics = {
    "precision": metrics.box.p.tolist(),
    "recall": metrics.box.r.tolist(),
    "f1": metrics.box.f1.tolist(),
    "ap50": metrics.box.ap50.tolist(),
    "ap": metrics.box.ap.tolist()
}

names = list(model.names.values())[:len(class_metrics["precision"])]  # align with actual detected classes
df = pd.DataFrame(class_metrics, index=names)

print("\n Per-class results:")
print(df)

# Create and plot Confusion Matrix
# Get predictions for test set
test_results = model.predict(str(TEST_IMAGES), save=False, verbose=False)

# Extract true and predicted labels
true_labels = []
pred_labels = []

for result in test_results:
    if result.boxes is not None:
        # Get ground truth (if available in result)
        if hasattr(result, 'boxes') and result.boxes.cls is not None:
            true_labels.extend(result.boxes.cls.cpu().numpy())

        # Get predictions
        if hasattr(result, 'boxes') and result.boxes.cls is not None:
            pred_labels.extend(result.boxes.cls.cpu().numpy())

# Create confusion matrix
if true_labels and pred_labels:
    cm = confusion_matrix(true_labels, pred_labels, labels=range(len(names)))

    # Plot confusion matrix
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - YOLOv8 VIN Character Detection', fontsize=16, pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Calculate and display accuracy metrics
    accuracy = np.trace(cm) / np.sum(cm) if np.sum(cm) > 0 else 0
    print(f"\n Overall Accuracy: {accuracy:.3f}")

    # Per-class accuracy
    class_accuracy = np.diag(cm) / np.sum(cm, axis=1)
    accuracy_df = pd.DataFrame({
        'Class': names,
        'Accuracy': class_accuracy,
        'Support': np.sum(cm, axis=1)
    })
    print("\n Per-class Accuracy:")
    print(accuracy_df)
else:
    print("Could not extract labels for confusion matrix")

# Additional performance metrics

print("\n Overall Metrics:")
print(f"mAP@0.5       : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.3f}")
print(f"Mean Precision: {metrics.box.mp:.3f}")
print(f"Mean Recall   : {metrics.box.mr:.3f}")

# Compute overall F1 score manually as mean of per-class F1
overall_f1 = float(np.mean(metrics.box.f1))
print(f"Overall F1    : {overall_f1:.3f}")

Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
Model summary (fused): 72 layers, 11,138,355 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 137.022.0 MB/s, size: 775.4 KB)
val: Scanning C:\JLR_VIN_2549\test\labels.cache... 160 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 160/160  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.2it/s 4.5s.4ss
                   all        160       3040      0.989      0.999      0.995      0.943
                     0        160        320      0.999          1      0.995      0.978
                     1        111        148      0.998          1      0.995      0.957
                     2        108        144      0.998          1      0.995      0.928
                     3         84        102      0.997          1      0.995      0.949
                     4        100        2

<Figure size 1600x1400 with 2 Axes>


 Overall Accuracy: 1.000

 Per-class Accuracy:
   Class  Accuracy  Support
0      0       1.0      320
1      1       1.0      149
2      2       1.0      144
3      3       1.0      102
4      4       1.0      216
5      5       1.0      109
6      6       1.0      134
7      7       1.0      205
8      8       1.0       61
9      9       1.0       61
10     A       1.0      162
11     B       1.0      446
12     C       1.0       77
13     D       NaN        0
14     E       NaN        0
15     F       1.0       32
16     G       1.0       68
17     H       NaN        0
18     J       NaN        0
19     K       NaN        0
20     L       1.0       85
21     M       1.0      176
22     P       1.0        1
23     R       NaN        0

 Overall Metrics:
mAP@0.5       : 0.995
mAP@0.5:0.95  : 0.943
Mean Precision: 0.989
Mean Recall   : 0.999
Overall F1    : 0.994


C:\Users\MURTI\AppData\Local\Temp\ipykernel_22164\4193104536.py:73: RuntimeWarning: invalid value encountered in divide
  class_accuracy = np.diag(cm) / np.sum(cm, axis=1)


In [ ]:
import os
import sys
import csv
import re
from pathlib import Path

import cv2
import numpy as np
from PySide6 import QtCore, QtGui, QtWidgets
from ultralytics import YOLO

# ============================================================================
# IMPORT CENTRALIZED CONFIG
# ============================================================================
from config import PROJECT_ROOT, YOLO_BEST_FIXED_SEED, CLASS_NAMES

#  CONFIG - Now uses centralized paths
DATASET_ROOT = str(PROJECT_ROOT)
MODEL_PATH   = str(YOLO_BEST_FIXED_SEED)

# class list (order MUST match training) - from config
NAMES = CLASS_NAMES

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".JPG",".PNG",".BMP",".TIF",".TIFF"}

# GT from filename
_VIN_PATTERNS = [
    r'\*[A-HJ-NPR-Z0-9]{17}\*',   # *VIN*
    r'[A-HJ-NPR-Z0-9]{17}',       # VIN
]
def extract_vin_from_filename(path: str) -> str | None:
    s = os.path.splitext(os.path.basename(path.upper()))[0]
    for p in _VIN_PATTERNS:
        m = re.search(p, s)
        if m:
            v = m.group(0)
            return v if v.startswith('*') else f"*{v}*"
    return None

# core helpers
def build_vin_from_result(ultra_result, keep_star=False, conf_thres=0.35, iou=0.5):
    boxes = ultra_result.boxes
    if boxes is None or len(boxes) == 0:
        return "", []
    items = []
    for b in boxes:
        cls = int(b.cls.item())
        ch  = NAMES[cls] if 0 <= cls < len(NAMES) else '?'
        if (not keep_star) and ch == '*':
            continue
        conf = float(b.conf.item())
        if conf < conf_thres:
            continue
        x1,y1,x2,y2 = [float(v) for v in b.xyxy[0].tolist()]
        xc = (x1+x2)/2.0
        items.append((xc, ch, conf, (x1,y1,x2,y2)))
    items.sort(key=lambda t: t[0])  # left-to-right
    vin = "".join(ch for _, ch, _, _ in items)
    boxes_out = [(x1,y1,x2,y2,ch,conf) for _, ch, conf, (x1,y1,x2,y2) in items]
    return vin, boxes_out

def draw_overlay(img_bgr, boxes, pred_vin, exp_vin=None, conf_used=0.35, low_conf_delta=0.10):
    canvas = img_bgr.copy()
    low_thr = max(0.0, conf_used - low_conf_delta)
    any_wrong = False

    for i, (x1,y1,x2,y2,ch,conf) in enumerate(boxes):
        correct = None
        if exp_vin is not None:
            if i < len(exp_vin):
                correct = (ch == exp_vin[i])
            else:
                correct = False
        # default (correct)
        color_box, color_text = (0,255,0), (0,0,255)
        if correct is False:
            color_box, color_text = (0,0,255), (255,255,255)
            any_wrong = True
        if conf < low_thr:  # low-confidence → orange
            color_box, color_text = (0,140,255), (255,255,255)

        p1 = (int(x1), int(y1)); p2 = (int(x2), int(y2))
        cv2.rectangle(canvas, p1, p2, color_box, 2)
        cv2.putText(canvas, ch, (p1[0], max(10, p1[1]-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_text, 2, cv2.LINE_AA)

    # headers
    y0 = 30
    title_ok = (exp_vin is not None and pred_vin == exp_vin and not any_wrong)
    title_color = (0,255,0) if title_ok else (0,0,255)
    cv2.putText(canvas, f"Pred: {pred_vin}", (10, y0),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, title_color, 2, cv2.LINE_AA)
    if exp_vin:
        cv2.putText(canvas, f"Exp : {exp_vin}", (10, y0+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2, cv2.LINE_AA)
    return canvas

def bgr2pixmap(img_bgr):
    h,w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    qimg = QtGui.QImage(img_rgb.data, w, h, 3*w, QtGui.QImage.Format_RGB888)
    return QtGui.QPixmap.fromImage(qimg)

# threaded predictor
class InferenceWorker(QtCore.QThread):
    progress = QtCore.Signal(int)                 # percent
    item_done = QtCore.Signal(dict)               # per-image result
    finished  = QtCore.Signal()

    def __init__(self, model: YOLO, paths, out_dir, conf, keep_star, parent=None):
        super().__init__(parent)
        self.model = model
        self.paths = paths
        self.out_dir = Path(out_dir) if out_dir else None
        self.conf = float(conf)
        self.keep_star = bool(keep_star)
        self._stop = False

    def stop(self):
        self._stop = True

    def run(self):
        csv_f, csv_w = None, None
        if self.out_dir:
            (self.out_dir / "annotated").mkdir(parents=True, exist_ok=True)
            csv_path = self.out_dir / "predictions.csv"
            csv_f = open(csv_path, "w", newline="", encoding="utf-8")
            csv_w = csv.writer(csv_f)
            csv_w.writerow(["image","expected","prediction","status"])

        n = max(1, len(self.paths))
        for i, path in enumerate(self.paths, start=1):
            if self._stop:
                break
            try:
                img = cv2.imread(str(path))
                if img is None:
                    raise RuntimeError(f"Failed to read: {path}")

                img_in = cv2.convertScaleAbs(img, alpha=1.2, beta=0)
                result = self.model.predict(img_in, conf=self.conf, iou=0.5, imgsz=1024, verbose=False)[0]
                pred_str, boxes = build_vin_from_result(result, keep_star=self.keep_star, conf_thres=self.conf)

                exp_vin = extract_vin_from_filename(path)
                overlay = draw_overlay(img, boxes, pred_str, exp_vin=exp_vin, conf_used=self.conf)

                status = "ok" if (exp_vin and pred_str == exp_vin) else ("wrong" if exp_vin else "unknown")
                if csv_w:
                    csv_w.writerow([Path(path).name, exp_vin or "", pred_str, status])
                    ann_path = self.out_dir / "annotated" / (Path(path).stem + "_pred.jpg")
                    cv2.imwrite(str(ann_path), overlay)

                self.item_done.emit({
                    "path": str(path),
                    "exp": exp_vin,
                    "pred": pred_str,
                    "status": status,
                    "overlay": overlay,
                    "boxes": boxes,
                })
            except Exception as e:
                self.item_done.emit({
                    "path": str(path),
                    "error": str(e),
                })
            self.progress.emit(int(100*i/n))

        if csv_f:
            csv_f.close()
        self.finished.emit()


# GUI

class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("VIN Inference GUI (Fixed-Seed Model)")
        self.resize(1280,800)
        self.model = None
        self._img_paths = []
        self._results = []
        self._idx = -1
        self._worker = None

        # layout
        central = QtWidgets.QWidget()
        self.setCentralWidget(central)
        vlay = QtWidgets.QVBoxLayout(central)

        # toolbar
        tbar = QtWidgets.QHBoxLayout()
        vlay.addLayout(tbar)

        self.btn_load = QtWidgets.QPushButton("Load Model")
        self.btn_load.clicked.connect(self._load_model)
        tbar.addWidget(self.btn_load)

        self.lbl_model = QtWidgets.QLabel("No model loaded")
        tbar.addWidget(self.lbl_model)
        tbar.addStretch()

        self.btn_folder = QtWidgets.QPushButton("Select Image Folder")
        self.btn_folder.clicked.connect(self._select_folder)
        tbar.addWidget(self.btn_folder)

        self.btn_run = QtWidgets.QPushButton("Run Inference")
        self.btn_run.clicked.connect(self._run_inference)
        self.btn_run.setEnabled(False)
        tbar.addWidget(self.btn_run)

        # conf slider
        tbar2 = QtWidgets.QHBoxLayout()
        vlay.addLayout(tbar2)
        tbar2.addWidget(QtWidgets.QLabel("Conf threshold:"))
        self.slider_conf = QtWidgets.QSlider(QtCore.Qt.Horizontal)
        self.slider_conf.setRange(10, 90)
        self.slider_conf.setValue(35)
        self.slider_conf.setTickInterval(5)
        self.slider_conf.setTickPosition(QtWidgets.QSlider.TicksBelow)
        tbar2.addWidget(self.slider_conf)
        self.lbl_conf = QtWidgets.QLabel("0.35")
        self.slider_conf.valueChanged.connect(lambda v: self.lbl_conf.setText(f"{v/100:.2f}"))
        tbar2.addWidget(self.lbl_conf)
        tbar2.addStretch()

        self.chk_star = QtWidgets.QCheckBox("Keep * char")
        self.chk_star.setChecked(False)
        tbar2.addWidget(self.chk_star)

        # progress
        self.progress = QtWidgets.QProgressBar()
        self.progress.setValue(0)
        vlay.addWidget(self.progress)

        # splitter: image | results table
        splitter = QtWidgets.QSplitter()
        vlay.addWidget(splitter, stretch=1)

        # image
        self.lbl_img = QtWidgets.QLabel("No image")
        self.lbl_img.setAlignment(QtCore.Qt.AlignCenter)
        self.lbl_img.setMinimumSize(640,480)
        splitter.addWidget(self.lbl_img)

        # table
        self.table = QtWidgets.QTableWidget(0, 4)
        self.table.setHorizontalHeaderLabels(["Image", "Expected", "Prediction", "Status"])
        self.table.horizontalHeader().setStretchLastSection(True)
        self.table.setSelectionBehavior(QtWidgets.QTableWidget.SelectRows)
        self.table.itemSelectionChanged.connect(self._on_table_select)
        splitter.addWidget(self.table)

        splitter.setSizes([700,500])

        # navigation
        nav = QtWidgets.QHBoxLayout()
        vlay.addLayout(nav)
        self.btn_prev = QtWidgets.QPushButton("← Prev")
        self.btn_prev.clicked.connect(self._prev_img)
        nav.addWidget(self.btn_prev)
        self.btn_next = QtWidgets.QPushButton("Next →")
        self.btn_next.clicked.connect(self._next_img)
        nav.addWidget(self.btn_next)
        nav.addStretch()
        self.btn_export = QtWidgets.QPushButton("Export CSV")
        self.btn_export.clicked.connect(self._export_csv)
        nav.addWidget(self.btn_export)

    # ─── slots ────────────────────────────────────────────────────────

    def _load_model(self):
        path, _ = QtWidgets.QFileDialog.getOpenFileName(self, "Select YOLO .pt", MODEL_PATH, "PyTorch (*.pt)")
        if not path:
            return
        try:
            self.model = YOLO(path)
            self.lbl_model.setText(f"Model: {Path(path).name}")
            self.btn_run.setEnabled(len(self._img_paths) > 0)
        except Exception as e:
            QtWidgets.QMessageBox.critical(self, "Error", str(e))

    def _select_folder(self):
        folder = QtWidgets.QFileDialog.getExistingDirectory(self, "Select image folder", DATASET_ROOT)
        if not folder:
            return
        self._img_paths = sorted([p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS])
        QtWidgets.QMessageBox.information(self, "Folder", f"Found {len(self._img_paths)} images")
        self.btn_run.setEnabled(self.model is not None and len(self._img_paths) > 0)

    def _run_inference(self):
        if self._worker and self._worker.isRunning():
            self._worker.stop()
            return
        self._results.clear()
        self.table.setRowCount(0)
        self.progress.setValue(0)

        out_dir, _ = QtWidgets.QFileDialog.getSaveFileName(self, "Output folder (pick any file name)", "", "Folder")
        if out_dir:
            out_dir = Path(out_dir).parent / (Path(out_dir).stem + "_results")
        else:
            out_dir = None

        conf = self.slider_conf.value() / 100.0
        keep_star = self.chk_star.isChecked()

        self._worker = InferenceWorker(self.model, self._img_paths, out_dir, conf, keep_star, self)
        self._worker.progress.connect(self.progress.setValue)
        self._worker.item_done.connect(self._on_item_done)
        self._worker.finished.connect(self._on_finished)
        self._worker.start()
        self.btn_run.setText("Stop")

    def _on_item_done(self, d):
        self._results.append(d)
        row = self.table.rowCount()
        self.table.insertRow(row)
        self.table.setItem(row, 0, QtWidgets.QTableWidgetItem(Path(d["path"]).name))
        self.table.setItem(row, 1, QtWidgets.QTableWidgetItem(d.get("exp", "")))
        self.table.setItem(row, 2, QtWidgets.QTableWidgetItem(d.get("pred", d.get("error", ""))))
        self.table.setItem(row, 3, QtWidgets.QTableWidgetItem(d.get("status", "error")))
        # auto-scroll
        self.table.scrollToBottom()

    def _on_finished(self):
        self.btn_run.setText("Run Inference")
        QtWidgets.QMessageBox.information(self, "Done", f"Processed {len(self._results)} images")

    def _on_table_select(self):
        rows = self.table.selectionModel().selectedRows()
        if not rows:
            return
        idx = rows[0].row()
        self._show_result(idx)

    def _show_result(self, idx):
        if idx < 0 or idx >= len(self._results):
            return
        self._idx = idx
        d = self._results[idx]
        if "overlay" in d:
            pm = bgr2pixmap(d["overlay"])
            self.lbl_img.setPixmap(pm.scaled(self.lbl_img.size(), QtCore.Qt.KeepAspectRatio, QtCore.Qt.SmoothTransformation))
        else:
            self.lbl_img.setText(d.get("error", "No image"))

    def _prev_img(self):
        if self._idx > 0:
            self._show_result(self._idx - 1)
            self.table.selectRow(self._idx)

    def _next_img(self):
        if self._idx < len(self._results) - 1:
            self._show_result(self._idx + 1)
            self.table.selectRow(self._idx)

    def _export_csv(self):
        path, _ = QtWidgets.QFileDialog.getSaveFileName(self, "Export CSV", "", "CSV (*.csv)")
        if not path:
            return
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["image","expected","prediction","status"])
            for d in self._results:
                w.writerow([Path(d["path"]).name, d.get("exp",""), d.get("pred",""), d.get("status","")])
        QtWidgets.QMessageBox.information(self, "Saved", f"CSV saved to {path}")


def run_gui():
    app = QtWidgets.QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())

if __name__ == "__main__":
    run_gui()

SystemExit: 0

c:\JLR_VIN_2549\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 